# Baseline de clasificacion de cultivos — Random Forest y XGBoost

Este notebook responde una pregunta concreta: **¿que tan lejos llega un modelo tabular sencillo para clasificar cultivos a partir de imagenes satelitales?** Se entrenan dos modelos de arboles (Random Forest y XGBoost) sobre un vector de caracteristicas que combina el embedding AlphaEarth de 64 dimensiones, indices espectrales, estadisticas temporales, terreno y clima.

El resultado sirve de **punto de referencia**: cualquier modelo mas complejo en fases posteriores tendra que superar estas cifras para justificar su coste.

## Requisitos para ejecucion end-to-end

- El subset PASTIS-R a nivel parcela descomprimido en `data/test_fixtures/`.
- Dependencias instaladas via `poetry install --with ml,geo`.

El notebook se ejecuta de principio a fin sin intervencion manual; los parametros (tamano de muestra, tuning) se ajustan desde la celda de parametros.

## Contenido

| Seccion | Contenido |
|---------|-----------|
| 1 | Carga del conjunto de datos |
| 2 | Por que Random Forest y XGBoost |
| 3 | Importancia de caracteristicas |
| 4 | Analisis SHAP |
| 5 | Conclusiones de ingenieria de caracteristicas |
| 5b | Curvas de aprendizaje y validacion |
| 6 | Desempeno del baseline |
| 7 | Comparativa AlphaEarth vs Sentinel-2 crudo |
| 8 | Conclusiones |


## 1. Carga del conjunto de datos

El conjunto de entrada es un subset de PASTIS-R a nivel de parcela: 85.951 parcelas agricolas con 187 caracteristicas espectro-temporales cada una. La etiqueta es el tipo de cultivo (20 clases de PASTIS-R; se descartan las clases de fondo).

In [1]:
# Parametros papermill (celda con tag 'parameters'; sobreescribibles
# en CI con valores reducidos via `papermill -p`).
FEATURES_PATH = 'data/test_fixtures/feature_selection_parcels_subset.parquet'
MAX_SAMPLES = 0  # 0 = dataset completo; >0 = submuestreo estratificado
TUNE = True
F1_THRESHOLD = 0.60
# Seccion 7 (US-022) — rutas de los 3 escenarios de la comparativa.
SCENARIO_ALPHAEARTH_PATH = (
    'data/cache/gee/alphaearth_pastis_parcels_2019_85951_enriched.parquet'
)
SCENARIO_S2_RAW_PATH = (
    'data/cache/pastis/s2_raw_parcels_2019_85951.parquet'
)
SCENARIO_COMBINED_PATH = (
    'data/test_fixtures/feature_selection_parcels_subset.parquet'
)
COMPARISON_MAX_SAMPLES = 0  # 0 = todas las parcelas del inner join
COMPARISON_K_FOLDS = 5


In [2]:
# Parameters
MAX_SAMPLES = 3000
TUNE = False
COMPARISON_MAX_SAMPLES = 3000


In [3]:
import warnings

import matplotlib

matplotlib.use('Agg')  # backend headless para papermill/CI
import matplotlib.pyplot as plt
import polars as pl

warnings.filterwarnings('ignore')


In [4]:
from ml.train.baseline import _load_baseline_dataset, _prepare_dataframe

df_raw = _load_baseline_dataset(FEATURES_PATH)
df = _prepare_dataframe(df_raw)
print(f'Parcelas: {df.height:,}  |  Columnas: {df.width}')
df.head()

Parcelas: 85,951  |  Columnas: 192


parcel_id,year,NDVI_mean,NDVI_std,NDVI_min,NDVI_max,NDVI_p05,NDVI_p25,NDVI_p50,NDVI_p75,NDVI_p95,NDWI_mean,NDWI_std,NDWI_min,NDWI_max,NDWI_p05,NDWI_p25,NDWI_p50,NDWI_p75,NDWI_p95,EVI_mean,EVI_std,EVI_min,EVI_max,EVI_p05,EVI_p25,EVI_p50,EVI_p75,EVI_p95,NDMI_mean,NDMI_std,NDMI_min,NDMI_max,NDMI_p05,NDMI_p25,NDMI_p50,NDMI_p75,…,NDVI_fft_amp_0,NDVI_fft_phase_0,NDVI_fft_amp_1,NDVI_fft_phase_1,NDVI_fft_amp_2,NDVI_fft_phase_2,NDVI_fft_amp_3,NDVI_fft_phase_3,NDWI_fft_amp_0,NDWI_fft_phase_0,NDWI_fft_amp_1,NDWI_fft_phase_1,NDWI_fft_amp_2,NDWI_fft_phase_2,NDWI_fft_amp_3,NDWI_fft_phase_3,EVI_fft_amp_0,EVI_fft_phase_0,EVI_fft_amp_1,EVI_fft_phase_1,EVI_fft_amp_2,EVI_fft_phase_2,EVI_fft_amp_3,EVI_fft_phase_3,sog_doy,peak_doy,peak_value,senescence_doy,ndvi_auc,ndvi_slope_pre_peak,ndvi_slope_post_peak,maturity_duration_days,patch_id,instance_id,class_id,fold,n_pixels
str,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,i64,f64,i64,f64,f64,f64,i64,i64,i64,i64,i64,i64
"""10000_1""",2018,0.451583,0.425313,-0.068311,2.303833,-0.02314,0.202424,0.30881,0.765301,0.987349,-0.4298,0.338675,-1.603543,0.203464,-0.822717,-0.656512,-0.421072,-0.231754,0.03352,0.282466,0.307185,-0.622353,0.848309,-0.13101,0.14015,0.189091,0.59712,0.721938,0.227215,0.304057,-0.235981,1.0,-0.225231,0.013784,0.166098,0.470995,…,0.406117,0.0,0.116952,0.959278,0.268396,-0.812251,0.099299,-0.671697,0.379628,0.0,0.096763,-1.491115,0.180904,2.22084,0.094876,1.680415,0.250078,0.0,0.096473,1.985061,0.109415,-1.297251,0.089628,-0.86289,null,26,2.303833,34,158.423406,null,-0.258612,5,10000,1,2,1,101
"""10000_2""",2018,0.499859,0.613426,-0.035138,3.852548,0.009661,0.16877,0.265388,0.779707,0.98985,-0.461037,0.451878,-2.664785,0.215764,-0.86394,-0.699827,-0.379332,-0.237233,0.039321,0.288321,0.351033,-1.282691,0.865093,-0.051102,0.124069,0.202867,0.622151,0.701521,0.260024,0.301983,-0.225433,1.0,-0.18454,0.048057,0.26025,0.490585,…,0.447687,0.0,0.119995,0.652629,0.360659,-0.887526,0.154474,-1.078157,0.406423,0.0,0.082969,-1.678393,0.250675,2.142489,0.150508,1.643163,0.263809,0.0,0.164197,2.412146,0.146805,-1.176721,0.016652,0.282209,null,26,3.852548,35,174.858654,null,-0.421472,4,10000,2,2,1,146
"""10000_3""",2018,0.334038,0.27952,-0.019577,1.0,-0.004163,0.14065,0.238016,0.552242,0.834303,-0.349186,0.319539,-0.8,1.030717,-0.777078,-0.604582,-0.335123,-0.228834,0.006794,0.213473,0.190922,-0.104255,0.618046,-0.023873,0.099734,0.138956,0.312178,0.585605,0.11725,0.250841,-0.331999,1.0,-0.162837,-0.064601,0.093857,0.233324,…,0.286841,0.0,0.174465,0.561766,0.163503,0.806981,0.101583,2.325232,0.298671,0.0,0.057297,-1.618627,0.054383,-2.954037,0.122988,0.383855,0.184555,0.0,0.09192,0.972171,0.106464,1.453878,0.081433,2.637839,184,366,1.0,377,111.748535,0.003212,-0.064142,24,10000,3,12,1,222
"""10000_5""",2018,0.38247,0.285132,-0.074386,1.0,0.022891,0.177489,0.318098,0.622669,0.806941,-0.394311,0.262345,-0.940892,0.184607,-0.788802,-0.603789,-0.385081,-0.232968,-0.023878,0.253364,0.197264,-0.182055,0.790065,0.052423,0.12536,0.178894,0.403373,0.580462,0.168672,0.248691,-0.272999,1.0,-0.209728,0.018661,0.152897,0.327187,…,0.35871,0.0,0.112893,2.013443,0.226917,-0.746715,0.054041,0.988047,0.360391,0.0,0.120399,-0.82425,0.168244,2.240376,0.037988,1.114151,0.240716,0.0,0.132097,2.513825,0.143179,-1.109168,0.019577,0.949213,42,366,1.0,377,139.890532,0.000525,-0.065199,5,10000,5,2,1,161
"""10000_7""",2018,0.31712,0.238965,0.001322,1.0,0.041599,0.180184,0.251987,0.3889,0.841654,-0.349337,0.234502,-1.0,0.264819,-0.67687,-0.491825,-0.353776,-0.195299,-0.025525,0.211746,0.151538,-0.036628,0.83954,0.041436,0.119197,0.188436,0.25756,0.496483,0.113138,0.267155,-0.26795,1.079542,-0.19123,-0.040898,0.078001,0.13962,…,0.277805,0.0,0.059054,1.143065,0.100011,-0.166592,0.098765,-0.294571,0.308596,0.0,0.077307,-0.308286,0.

In [5]:
# Distribucion de clases — PASTIS-R tiene desbalance fuerte.
class_counts = (
    df.group_by('class_id').len().sort('len', descending=True)
)
class_counts

class_id,len
i64,u32
1,31292
3,13123
8,10640
2,8206
14,3174
…,…
6,908
9,871
17,848


## 2. Por que Random Forest y XGBoost

Se eligen **Random Forest** y **XGBoost** como modelos de referencia. Cuatro razones sustentan la decision:

**(a) Las imagenes ya vienen resumidas.** El embedding AlphaEarth de 64 dimensiones condensa informacion optica, radar y temporal aprendida por un modelo entrenado sobre todo el archivo Sentinel. Sobre una representacion ya rica, un modelo de arboles es un punto de referencia suficiente y honesto — no hace falta una red neuronal profunda para establecer el piso de desempeno (cf. Brown et al., 2025, *AlphaEarth Foundations*).

**(b) Son interpretables.** Ambos exponen una medida de importancia de caracteristicas (Gini para Random Forest, *gain* para XGBoost) y son compatibles con SHAP. Esto permite auditar que variables explican las predicciones — un modelo opaco no lo permitiria (Lundberg & Lee, 2017, *SHAP*).

**(c) Son robustos a valores atipicos y a la escala.** Los arboles dividen el espacio por umbrales y no asumen ninguna distribucion de las variables; los valores atipicos residuales no desplazan las fronteras de decision como lo harian en un modelo lineal sin normalizacion cuidadosa.

**(d) Tienen bajo coste computacional.** El problema (85.951 parcelas, 187 variables, 20 clases) se entrena en minutos. XGBoost aprovecha la GPU local cuando esta disponible y degrada a CPU de forma transparente; Random Forest corre siempre en CPU multinucleo. El experimento es reproducible en cualquier laptop.

## 3. Importancia de caracteristicas

Random Forest y XGBoost exponen una medida de importancia de caracteristicas sin coste adicional: **Gini** para Random Forest y **gain** para XGBoost. Es el primer diagnostico de interpretabilidad — barato y directo — antes del analisis SHAP de la seccion 4.

Se cargan los modelos ya entrenados desde `artifacts/baseline_{rf,xgb}_v1.joblib`; si los archivos no existen, el notebook entrena los modelos en el momento con los hiperparametros base.

In [6]:
import joblib
from pathlib import Path

from ml.train.baseline import train_one_model

REPORTS_DIR = Path('reports/baseline')
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS = {'rf': Path('artifacts/baseline_rf_v1.joblib'),
             'xgb': Path('artifacts/baseline_xgb_v1.joblib')}

models = {}
for kind, path in ARTIFACTS.items():
    if path.exists():
        payload = joblib.load(path)
        models[kind] = {
            'model': payload['model'],
            'feature_cols': tuple(payload['feature_cols']),
            'source': 'joblib US-019',
        }
    else:
        res = train_one_model(df, model=kind)
        models[kind] = {
            'model': res.model,
            'feature_cols': res.feature_cols,
            'source': 'fallback in-notebook (D8)',
        }
    print(f"{kind.upper()}: {models[kind]['source']}  |  "
          f"{len(models[kind]['feature_cols'])} features")

RF: joblib US-019  |  185 features


XGB: joblib US-019  |  185 features


In [7]:
from ml.eval.interpretability import feature_importance_table

importance = {}
for kind, bundle in models.items():
    table = feature_importance_table(
        bundle['model'], kind, bundle['feature_cols']
    )
    importance[kind] = table
    table.write_csv(REPORTS_DIR / f'feature_importance_{kind}.csv')
importance['rf'].head(10)

2026-05-22 10:41:21 [info     ] feature_importance_table_computed model_kind=rf n_features=185 top_feature=EVI_fft_phase_2


2026-05-22 10:41:22 [info     ] feature_importance_table_computed model_kind=xgb n_features=185 top_feature=MSAVI2_min


feature,importance,rank
str,f64,i64
"""EVI_fft_phase_2""",0.031686,1
"""CCCI_p75""",0.025438,2
"""EVI_fft_phase_1""",0.024245,3
"""EVI_fft_phase_3""",0.020832,4
"""NDCI_p50""",0.019787,5
"""MCARI_p95""",0.019077,6
"""NDVI_fft_phase_2""",0.018781,7
"""NDVI_p50""",0.018475,8
"""NDWI_fft_phase_1""",0.018447,9


In [8]:
# Barplot top-20 de la importancia nativa por modelo.
for kind, table in importance.items():
    top20 = table.head(20)
    fig, ax = plt.subplots(figsize=(8, 6), dpi=200)
    ax.barh(top20['feature'].to_list()[::-1],
            top20['importance'].to_list()[::-1],
            color='#2c7fb8')
    ax.set_xlabel('Importancia (' + ('Gini' if kind == 'rf' else 'gain') + ')')
    ax.set_title(f'Importancia nativa top-20 — {kind.upper()}')
    fig.tight_layout()
    fig.savefig(REPORTS_DIR / f'importance_{kind}_top20.png',
                dpi=200, bbox_inches='tight')
    plt.show()

## 4. Analisis SHAP

La importancia de la seccion 3 ordena las caracteristicas pero no explica *como* cada una desplaza la prediccion. **SHAP** (Lundberg & Lee, 2017) descompone cada prediccion en contribuciones aditivas por caracteristica, con garantias teoricas de consistencia. Para modelos de arboles se usa el algoritmo TreeSHAP, que es exacto.

Detalles de la implementacion:

- **Submuestreo**: SHAP se calcula sobre una muestra estratificada de ~3.000 parcelas, no sobre las ~85.000 del conjunto; el coste de TreeSHAP crece con el numero de muestras, arboles y profundidad.
- **Multiclase**: PASTIS-R tiene 18-20 clases; la salida multiclase de SHAP se normaliza a un tensor uniforme `(muestras, caracteristicas, clases)`.
- **Ranking global**: la importancia global es el promedio del valor absoluto de SHAP sobre clases y muestras.

In [9]:
from ml.eval.interpretability import (
    compute_shap_values,
    shap_summary_plot,
    shap_dependence_plots,
    shap_waterfall_plot,
)

SHAP_SAMPLE_SIZE = 3000
shap_results = {}
for kind, bundle in models.items():
    shap_results[kind] = compute_shap_values(
        bundle['model'], df, kind,
        feature_cols=bundle['feature_cols'],
        sample_size=SHAP_SAMPLE_SIZE,
    )
    print(f'{kind.upper()}: tensor SHAP '
          f'{shap_results[kind].values.shape}')

2026-05-22 10:52:54 [info     ] shap_values_computed           model_kind=rf n_classes=18 n_features=185 n_samples=3000 sample_rows=3000


RF: tensor SHAP (3000, 185, 18)


2026-05-22 10:53:02 [info     ] shap_values_computed           model_kind=xgb n_classes=18 n_features=185 n_samples=3000 sample_rows=3000


XGB: tensor SHAP (3000, 185, 18)


In [10]:
# Summary plot (beeswarm/bar) de las top-20 features globales.
for kind, result in shap_results.items():
    fig = shap_summary_plot(result, df, top_n=20)
    fig.savefig(REPORTS_DIR / f'shap_summary_{kind}.png',
                dpi=200, bbox_inches='tight')
    plt.show()

In [11]:
# Dependence plots de los 5 features mas importantes (RF).
dependence = shap_dependence_plots(
    shap_results['rf'], df, top_features=5
)
for idx, (feature_name, fig) in enumerate(dependence, start=1):
    fig.savefig(
        REPORTS_DIR / f'shap_dependence_{idx}_{feature_name}.png',
        dpi=200, bbox_inches='tight',
    )
    plt.show()

2026-05-22 10:53:05 [info     ] shap_dependence_plots_generated class_idx=0 model_kind=rf n_plots=5


In [12]:
# Waterfall de una prediccion ejemplo por modelo.
for kind, result in shap_results.items():
    fig = shap_waterfall_plot(result, row=0)
    fig.savefig(REPORTS_DIR / f'shap_waterfall_{kind}.png',
                dpi=200, bbox_inches='tight')
    plt.show()

### 4.1 Dominancia de las dimensiones AlphaEarth

Una pregunta interesante: de las caracteristicas mas influyentes segun SHAP, **¿cuantas son dimensiones del embedding AlphaEarth** (`dim_00..dim_63`) frente a indices espectrales, estadisticas temporales o bloques de contexto (radar, terreno, clima)? La respuesta indica cuanto del poder predictivo proviene del embedding satelital frente al resto de las caracteristicas.

In [13]:
from ml.eval.interpretability import alphaearth_dominance_table

dominance = alphaearth_dominance_table(
    shap_results['rf'].global_importance, top_n=20
)
dominance.write_csv(REPORTS_DIR / 'alphaearth_dominance.csv')
dominance

2026-05-22 10:53:07 [info     ] alphaearth_dominance_computed  dominance_ratio=0.0 n_alphaearth=0 top_n=20


rank,feature,family,importance
i64,str,str,f64
1,"""EVI_fft_phase_2""","""spectral_index""",0.004859
2,"""CCCI_p75""","""spectral_index""",0.004565
3,"""EVI_fft_phase_1""","""spectral_index""",0.004053
4,"""MCARI_p95""","""spectral_index""",0.003342
5,"""NDVI_fft_phase_2""","""spectral_index""",0.003161
…,…,…,…
16,"""NDRE_p50""","""spectral_index""",0.002208
17,"""GCVI_p50""","""spectral_index""",0.002186
18,"""EVI_p95""","""spectral_index""",0.002094


In [14]:
# Conteo por familia y conclusion cuantificada.
family_counts = (
    dominance.group_by('family').len()
    .sort('len', descending=True)
)
n_alphaearth = int(
    dominance.filter(pl.col('family') == 'alphaearth').height
)
top_ae = (
    dominance.filter(pl.col('family') == 'alphaearth')['feature']
    .to_list()[:3]
)
print(f'{n_alphaearth}/20 de las top features SHAP son '
      f'dimensiones AlphaEarth.')
if top_ae:
    print(f'Lideran: ' + ', '.join(top_ae))
family_counts

0/20 de las top features SHAP son dimensiones AlphaEarth.


family,len
str,u32
"""spectral_index""",20


## 5. Conclusiones de ingenieria de caracteristicas

Esta seccion **valida o cuestiona** las decisiones de ingenieria de caracteristicas de la fase anterior, cruzando los rankings de interpretabilidad de este notebook con los resultados de la seleccion de variables previa:

- `reports/feature_selection/feature_importance_rf.csv` — importancia exploratoria de la fase de seleccion.
- `reports/feature_selection/anova_f_scores.csv` — F-scores univariados de la seleccion.
- `reports/feature_selection/selected_features.json` — el conjunto de variables que se retuvo.

El objetivo es responder tres preguntas: (a) ¿las caracteristicas mas influyentes segun SHAP coinciden con las que se seleccionaron?; (b) ¿alguna variable descartada aparece como importante?; (c) ¿la dominancia de AlphaEarth confirma la decision de usar el embedding como base?

In [15]:
# Cruce de las top SHAP con la seleccion de variables previa.
fs_dir = Path('reports/feature_selection')
top_shap = set(
    shap_results['rf'].global_importance.head(20)['feature'].to_list()
)

fs_importance_path = fs_dir / 'feature_importance_rf.csv'
if fs_importance_path.exists():
    fs_importance = pl.read_csv(fs_importance_path)
    fs_top = set(fs_importance.head(20)['feature'].to_list())
    overlap = top_shap & fs_top
    print(f'Solapamiento top-20 SHAP vs seleccion previa: '
          f'{len(overlap)}/20 caracteristicas.')
    print('Comunes:', sorted(overlap))
    print('Solo en SHAP (revisar FE):', sorted(top_shap - fs_top))
else:
    print('reports/feature_selection/feature_importance_rf.csv '
          'no disponible — se omite el cruce cuantitativo.')

Solapamiento top-20 SHAP vs seleccion previa: 10/20 caracteristicas.
Comunes: ['EVI_fft_phase_1', 'EVI_fft_phase_2', 'EVI_fft_phase_3', 'EVI_p95', 'NDRE_p95', 'NDVI_fft_phase_1', 'NDVI_fft_phase_2', 'NDVI_p50', 'NDWI_fft_phase_1', 'PSRI_p95']
Solo en SHAP (revisar FE): ['CCCI_p75', 'EVI_p25', 'GCVI_p50', 'MCARI_p25', 'MCARI_p50', 'MCARI_p95', 'MSAVI2_min', 'NDCI_p50', 'NDRE_p50', 'NDWI_p50']


### 5.1 Hallazgos

Los numeros concretos del cruce salen de la celda anterior. Los hallazgos que cabe esperar:

1. **Coincidencia entre la importancia simple y SHAP** — las caracteristicas en lo alto del ranking de Gini/gain y las del ranking SHAP coinciden en su mayoria; las discrepancias señalan variables con efectos no lineales o interacciones que SHAP captura mejor que la importancia simple.
2. **Dominancia de AlphaEarth** — la fraccion de dimensiones del embedding (`dim_NN`) entre las 20 mas influyentes (seccion 4.1) indica cuanto del poder predictivo proviene del embedding: si dominan, aporta la mayor parte de la senal; si no, los indices espectrales y las estadisticas estacionales siguen siendo imprescindibles.
3. **Validacion de la seleccion de variables** — si las caracteristicas seleccionadas en la fase previa coinciden con el top de SHAP, la seleccion queda validada; si una variable descartada aparece arriba, es una señal de que conviene revisarla.

### 5.2 Recomendacion para la ingenieria de caracteristicas

Si el cruce de la seccion 5 confirma la seleccion previa, **no se requiere ajuste**: la interpretabilidad del baseline la respalda. Si el cruce cuestiona alguna decision (una variable relevante descartada, o ruido retenido entre las mas influyentes), la recomendacion concreta se documenta para que las fases siguientes la incorporen antes de entrenar modelos mas complejos.

## 5b. Curvas de aprendizaje y validacion — diagnostico de sub/sobreajuste

Esta seccion diagnostica si el baseline sub o sobreajusta. Se usan dos herramientas:

- **Curva de aprendizaje**: accuracy de train y de validacion al crecer el numero de muestras de entrenamiento. Un gap grande train-val indica sobreajuste; ambas curvas bajas y juntas, subajuste.
- **Curva de validacion**: accuracy frente a un hiperparametro critico (`max_depth` para RF, `n_estimators` y `learning_rate` para XGBoost), para localizar la zona de equilibrio.

Toda la evaluacion usa el **mismo CV espacial 5-fold** (H3 + KMeans + buffer 1 km) del resto del notebook — los splits se materializan en una lista porque `learning_curve` reusa el `cv` una vez por cada tamano. El criterio de spatial CV esta documentado en `docs/spatial_cv_baseline.md`.

In [16]:
from ml.eval.learning_curves import (
    diagnose_fit,
    plot_learning_curve,
    plot_validation_curve,
)
from ml.train.baseline import _build_cv_splits

# CV espacial materializado (lista de splits posicionales).
cv_splits_5b = _build_cv_splits(
    df, k_folds=5, buffer_km=1.0, random_state=42
)
print(f'CV espacial: {len(cv_splits_5b)} folds materializados')

2026-05-22 10:53:07 [info     ] spatial_folds_building         buffer_km=1.0 k_folds=5 n_rows=85951 note='O(N^2) — puede tardar minutos en datasets grandes'


2026-05-22 10:53:42 [info     ] spatial_kfold_built            buffer_km=1.0 effective_k=5 excluded=0 h3_res=5 k=5 n_parcels=85951 n_unique_h3=218


2026-05-22 10:53:43 [info     ] spatial_folds_cache_saved      n_folds=5 path=data\test_fixtures\baseline_spatial_folds_n85951_k5_b1_s42.parquet


CV espacial: 5 folds materializados


In [17]:
# Curva de aprendizaje RF y XGB (accuracy train/val vs n muestras).
from pathlib import Path

from ml.train.baseline import build_estimator

reports_dir = Path('reports/baseline')
reports_dir.mkdir(parents=True, exist_ok=True)
curve_train_sizes = [0.1, 0.25, 0.4, 0.55, 0.7, 0.85, 1.0]
learning_results = {}
for kind in ('rf', 'xgb'):
    estimator = build_estimator(kind, {})
    lc_result, lc_fig = plot_learning_curve(
        estimator, df, cv_splits_5b,
        train_sizes=curve_train_sizes,
        max_samples=MAX_SAMPLES,
    )
    learning_results[kind] = lc_result
    lc_fig.suptitle(f'Curva de aprendizaje — {kind.upper()}')
    lc_fig.savefig(
        reports_dir / f'learning_curve_{kind}.png',
        dpi=200, bbox_inches='tight',
    )
    plt.show()

2026-05-22 10:53:43 [info     ] learning_curve_subsampled      max_samples=3000 n_kept=2999 n_original=85951


2026-05-22 10:53:43 [info     ] learning_curve_start           n_features=185 n_folds=5 n_samples=2999 n_train_sizes=7 scoring=accuracy


2026-05-22 10:54:28 [info     ] learning_curve_done            train_acc_max=1.0 val_acc_max=0.6766


2026-05-22 10:54:28 [info     ] xgb_device_resolved            device=cuda gpu='NVIDIA GeForce RTX 4070 Laptop GPU'


2026-05-22 10:54:28 [info     ] learning_curve_subsampled      max_samples=3000 n_kept=2999 n_original=85951


2026-05-22 10:54:28 [info     ] learning_curve_start           n_features=185 n_folds=5 n_samples=2999 n_train_sizes=7 scoring=accuracy


2026-05-22 10:55:48 [info     ] learning_curve_done            train_acc_max=1.0 val_acc_max=0.6974


In [18]:
# Diagnostico explicito de sub/sobreajuste por modelo.
for kind, lc_result in learning_results.items():
    diag = diagnose_fit(lc_result)
    print(f'{kind.upper()}: veredicto={diag.verdict}  '
          f'gap={diag.gap:.4f}  '
          f'train_acc={diag.train_acc_max:.4f}  '
          f'val_acc={diag.val_acc_max:.4f}')
    print(f'  {diag.explanation}')

2026-05-22 10:55:48 [info     ] fit_diagnosed                  gap=0.3234 train_acc_max=1.0 val_acc_max=0.6766 verdict=overfit


RF: veredicto=overfit  gap=0.3234  train_acc=1.0000  val_acc=0.6766
  Sobreajuste: el gap train-val es 0.323 (> 0.10). El modelo memoriza el train (accuracy 1.000) pero generaliza peor en validacion (accuracy 0.677).
2026-05-22 10:55:48 [info     ] fit_diagnosed                  gap=0.3026 train_acc_max=1.0 val_acc_max=0.6974 verdict=overfit


XGB: veredicto=overfit  gap=0.3026  train_acc=1.0000  val_acc=0.6974
  Sobreajuste: el gap train-val es 0.303 (> 0.10). El modelo memoriza el train (accuracy 1.000) pero generaliza peor en validacion (accuracy 0.697).


In [19]:
# Curva de validacion RF — max_depth.
vc_rf, vc_rf_fig = plot_validation_curve(
    build_estimator('rf', {}), df, 'max_depth',
    [5, 10, 15, 20, 30, None], cv_splits_5b,
    max_samples=MAX_SAMPLES,
)
vc_rf_fig.suptitle('Curva de validacion — RF max_depth')
vc_rf_fig.savefig(
    reports_dir / 'validation_curve_rf_max_depth.png',
    dpi=200, bbox_inches='tight',
)
plt.show()

2026-05-22 10:55:49 [info     ] learning_curve_subsampled      max_samples=3000 n_kept=2999 n_original=85951


2026-05-22 10:55:49 [info     ] validation_curve_start         n_folds=5 n_samples=2999 n_values=6 param_name=max_depth scoring=accuracy


2026-05-22 10:56:48 [info     ] validation_curve_done          best_val_acc=0.6761 param_name=max_depth


In [20]:
# Curva de validacion XGB — n_estimators.
vc_xgb_ne, vc_xgb_ne_fig = plot_validation_curve(
    build_estimator('xgb', {}), df, 'n_estimators',
    [100, 200, 300, 400, 500], cv_splits_5b,
    max_samples=MAX_SAMPLES,
)
vc_xgb_ne_fig.suptitle('Curva de validacion — XGB n_estimators')
vc_xgb_ne_fig.savefig(
    reports_dir / 'validation_curve_xgb_n_estimators.png',
    dpi=200, bbox_inches='tight',
)
plt.show()

2026-05-22 10:56:48 [info     ] xgb_device_resolved            device=cuda gpu='NVIDIA GeForce RTX 4070 Laptop GPU'


2026-05-22 10:56:49 [info     ] learning_curve_subsampled      max_samples=3000 n_kept=2999 n_original=85951


2026-05-22 10:56:49 [info     ] validation_curve_start         n_folds=5 n_samples=2999 n_values=5 param_name=n_estimators scoring=accuracy


2026-05-22 11:00:26 [info     ] validation_curve_done          best_val_acc=0.6939 param_name=n_estimators


El diagnostico reporta un veredicto explicito (sobreajuste, subajuste o ajuste adecuado) con la diferencia numerica entre el desempeno en entrenamiento y en validacion. Un modelo de arboles sobre estas caracteristicas tiende a una exactitud modesta: si el veredicto es *ajuste adecuado* pero con exactitud de validacion baja, el limite es la **capacidad del modelo**, no el sobreajuste — esto justifica que las fases siguientes incorporen arquitecturas temporales con mayor capacidad.

## 6. Desempeno del baseline

Se define un umbral de referencia de **F1-macro >= 0.60** sobre PASTIS-R. Se entrenan Random Forest y XGBoost con validacion cruzada **espacial** (celdas hexagonales H3 + agrupamiento KMeans + zona de exclusion de 1 km, para que parcelas vecinas no queden a la vez en entrenamiento y validacion) y se reporta el promedio de cada metrica sobre los pliegues.

Lo importante es que el desempeno quede **medido y explicado**: si el F1-macro no alcanza 0.60, la seccion 6.1 documenta las causas probables y las decisiones para las fases siguientes.

In [21]:
from ml.train.baseline import train_one_model, tune_baseline

results = {}
for kind in ('rf', 'xgb'):
    if TUNE:
        best_params = tune_baseline(df, model=kind)
        results[kind] = train_one_model(
            df, model=kind, hyperparams=best_params
        )
    else:
        results[kind] = train_one_model(df, model=kind)
    print(f'{kind.upper()}  entrenado.')

2026-05-22 11:00:26 [info     ] spatial_folds_cache_hit        n_folds=5 path=data\test_fixtures\baseline_spatial_folds_n85951_k5_b1_s42.parquet


2026-05-22 11:00:26 [info     ] spatial_cv_start               n_classes=18 n_folds=5


2026-05-22 11:00:26 [info     ] spatial_cv_fold_start          fold=1/5 n_test=23157 n_train=62794


2026-05-22 11:00:26 [info     ] scaler_persisted               n_features=185 n_train=62794 path=C:\Users\arthu\AppData\Local\Temp\tmpl5ky5lr4\fold_0_scaler.joblib version=v1


2026-05-22 11:00:44 [info     ] spatial_cv_fold_done           f1_macro=0.3895 fold=1/5


...

2026-05-22 11:10:11 [info     ] xgb_device_resolved            device=cuda gpu='NVIDIA GeForce RTX 4070 Laptop GPU'


2026-05-22 11:11:38 [info     ] spatial_cv_fold_done           f1_macro=0.3125 fold=5/5


2026-05-22 11:11:38 [info     ] xgb_device_resolved            device=cuda gpu='NVIDIA GeForce RTX 4070 Laptop GPU'


2026-05-22 11:13:00 [info     ] baseline_trained               f1_macro_oof=0.4094206066320682 model=xgb n_classes=18 n_features=185 n_samples=85951


XGB  entrenado.


In [22]:
# Tabla resumen de las metricas CV-mean por modelo.
summary = pl.DataFrame(
    [
        {
            'modelo': kind.upper(),
            **{m: round(v, 4) for m, v in res.metrics.items()},
        }
        for kind, res in results.items()
    ]
)
summary

modelo,f1_macro,f1_weighted,miou,accuracy,cohen_kappa
str,f64,f64,f64,f64,f64
"""RF""",0.365,0.6583,0.2699,0.6724,0.5961
"""XGB""",0.4094,0.6917,0.3115,0.7257,0.6546


In [23]:
# Veredicto frente al umbral de referencia.
best_kind = max(results, key=lambda k: results[k].metrics['f1_macro'])
best_f1 = results[best_kind].metrics['f1_macro']
passed = best_f1 >= F1_THRESHOLD
print(f'Mejor modelo: {best_kind.upper()}  |  F1-macro = {best_f1:.4f}')
print(f'Umbral de referencia: {F1_THRESHOLD:.2f}  |  '
      f'{"alcanzado" if passed else "no alcanzado — ver 6.1"}')

Mejor modelo: XGB  |  F1-macro = 0.4094
Umbral de referencia: 0.60  |  no alcanzado — ver 6.1


### 6.1 Causas probables y decisiones para las fases siguientes

Si el F1-macro promedio queda por debajo de 0.60, las causas probables son:

1. **Gran cantidad de clases (20 tipos de cultivo).** Varios cultivos son espectralmente parecidos; un modelo de arboles sobre un resumen anual no capta la firma estacional que los distingue.
2. **Clases desbalanceadas.** Pese al balanceo aplicado, las clases minoritarias aportan pocas parcelas y el F1-macro las penaliza con fuerza.
3. **Limite de un modelo de arboles sobre un resumen anual.** El embedding AlphaEarth condensa el ano en 64 dimensiones y pierde la dinamica intra-anual que un modelo de series temporales si aprovecha.

Decisiones concretas para las fases siguientes:

- Modelos que explotan la **serie temporal completa** de Sentinel-2 (no el resumen anual), capaces de captar la estacionalidad que separa cultivos parecidos.
- **Combinar varios modelos** (de arboles, temporales y de lenguaje-vision) para recuperar senal complementaria que ningun modelo aislado captura.

## 7. Comparativa AlphaEarth vs Sentinel-2 crudo

Esta seccion compara el baseline sobre **tres vistas distintas de las mismas parcelas**, para responder con evidencia una pregunta central: ¿el embedding AlphaEarth aporta valor frente a las bandas Sentinel-2 sin procesar?

| Escenario | Caracteristicas | Origen |
|-----------|-----------------|--------|
| **(a) AlphaEarth** | 64 dimensiones | embedding AlphaEarth Foundations |
| **(b) Sentinel-2 crudo** | 10 bandas promedio | bandas Sentinel-2 sin procesar, agregadas por parcela |
| **(c) Vector combinado** | 187 caracteristicas | ingenieria de caracteristicas espectro-temporales |

Metodologia de la comparativa:

- Los 3 escenarios se cruzan por parcela para evaluarse sobre **exactamente el mismo conjunto de parcelas**, no sobre tres muestras distintas.
- Se reutiliza la **misma validacion cruzada espacial** para los 3 escenarios; asi la diferencia de F1-macro refleja la calidad de las caracteristicas, no el azar de la particion.
- Se reporta tambien el **tiempo de entrenamiento** de cada modelo.

Si el escenario (b) Sentinel-2 crudo aun no se ha generado, esta seccion degrada de forma controlada y documenta la ausencia sin interrumpir el notebook.

In [24]:
from pathlib import Path

from ml.eval.comparison import (
    build_comparison_table,
    export_comparison_latex,
)

scenario_paths = {
    'alphaearth': SCENARIO_ALPHAEARTH_PATH,
    's2_raw': SCENARIO_S2_RAW_PATH,
    'combined': SCENARIO_COMBINED_PATH,
}
missing = {
    key: path
    for key, path in scenario_paths.items()
    if not Path(path).exists()
}
comparison_available = not missing
if missing:
    print('Escenarios no disponibles -> comparativa omitida:')
    for key, path in missing.items():
        print(f'  - {key}: {path}')
    print('Genera el escenario (b) con `make s2-raw-parcels`.')
else:
    print('Los 3 escenarios estan disponibles para la comparativa.')

Los 3 escenarios estan disponibles para la comparativa.


In [25]:
# Comparativa de los 3 escenarios (6 filas = 3 escenarios x 2 modelos).
comparison_result = None
if comparison_available:
    comparison_result = build_comparison_table(
        scenario_paths,
        k_folds=COMPARISON_K_FOLDS,
        max_samples=COMPARISON_MAX_SAMPLES,
        random_state=42,
    )
    print(f'Parcelas en el inner join: '
          f'{comparison_result.n_parcels:,}')
    comparison_result.table
else:
    print('Comparativa omitida — ver celda anterior.')

2026-05-22 11:13:01 [info     ] scenarios_aligned              n_common=85951 scenarios=['alphaearth', 'combined', 's2_raw']


2026-05-22 11:13:01 [info     ] comparison_subsampled          max_samples=3000 n_parcels=2999


2026-05-22 11:13:01 [info     ] comparison_table_start         k_folds=5 n_effective=2999 n_parcels=85951


2026-05-22 11:13:01 [info     ] spatial_folds_building         buffer_km=1.0 k_folds=5 n_rows=2999 note='O(N^2) — puede tardar minutos en datasets grandes'


2026-05-22 11:13:02 [info     ] spatial_kfold_built            buffer_km=1.0 effective_k=5 excluded=0 h3_res=5 k=5 n_parcels=2999 n_unique_h3=214


...

2026-05-22 11:19:22 [info     ] xgb_device_resolved            device=cuda gpu='NVIDIA GeForce RTX 4070 Laptop GPU'


2026-05-22 11:19:48 [info     ] baseline_trained               f1_macro_oof=0.30812850081705606 model=xgb n_classes=18 n_features=185 n_samples=2999


2026-05-22 11:19:48 [info     ] comparison_cell_done           f1_macro=0.3081 model=xgb scenario=combined train_time_s=142.3


2026-05-22 11:19:48 [info     ] comparison_table_done          alphaearth_delta=0.132 best_scenario='AlphaEarth 64-dim' n_parcels=85951


Parcelas en el inner join: 85,951


In [26]:
# Persistencia de la tabla comparativa (CSV + MD + LaTeX).
if comparison_result is not None:
    reports_dir = Path('reports/baseline')
    reports_dir.mkdir(parents=True, exist_ok=True)
    comparison_result.table.write_csv(
        reports_dir / 'comparison_alphaearth_vs_s2.csv'
    )
    md_table = (
        '# Comparativa de escenarios — baseline de cultivos\n\n'
        + comparison_result.table.to_pandas().to_markdown(index=False)
        + '\n'
    )
    (reports_dir / 'comparison_alphaearth_vs_s2.md').write_text(
        md_table, encoding='utf-8'
    )
    tex_path = export_comparison_latex(
        comparison_result, reports_dir / 'comparison_table.tex'
    )
    print(f'Tabla comparativa escrita: CSV + MD + {tex_path.name}')
else:
    print('Sin tabla comparativa que persistir.')

2026-05-22 11:19:48 [info     ] comparison_latex_written       path=reports\baseline\comparison_table.tex


Tabla comparativa escrita: CSV + MD + comparison_table.tex


In [27]:
# Barplot comparativo de F1-macro por escenario y modelo.
if comparison_result is not None:
    table = comparison_result.table
    scenarios = table['scenario'].unique(maintain_order=True).to_list()
    x = range(len(scenarios))
    width = 0.38
    fig, ax = plt.subplots(figsize=(9, 5), dpi=200)
    for offset, model in zip((-width / 2, width / 2), ('RF', 'XGB')):
        f1_by_scenario = [
            float(
                table.filter(
                    (pl.col('scenario') == sc)
                    & (pl.col('model') == model)
                )['f1_macro'][0]
            )
            for sc in scenarios
        ]
        bars = ax.bar(
            [xi + offset for xi in x], f1_by_scenario,
            width=width, label=model,
        )
        ax.bar_label(bars, fmt='%.3f', fontsize=8, padding=2)
    ax.set_xticks(list(x))
    ax.set_xticklabels(scenarios, rotation=15, ha='right', fontsize=9)
    ax.set_ylabel('F1-macro (CV espacial out-of-fold)')
    ax.set_ylim(0.0, 1.0)
    ax.set_title('Comparativa del baseline — 3 escenarios de features')
    ax.legend(title='Modelo')
    ax.grid(axis='y', alpha=0.3)
    fig.tight_layout()
    fig.savefig(
        Path('reports/baseline') / 'comparison_barplot.png',
        dpi=200, bbox_inches='tight',
    )
    plt.show()
else:
    print('Sin barplot — comparativa omitida.')

In [28]:
# Resumen cuantitativo del valor incremental de AlphaEarth.
if comparison_result is not None:
    delta = comparison_result.alphaearth_delta
    print(f'Escenario ganador: {comparison_result.best_scenario}')
    print(f'Delta F1-macro AlphaEarth - Sentinel-2 crudo: '
          f'{delta:+.4f}')
    if delta > 0.0:
        print('-> El embedding AlphaEarth aporta valor incremental '
              'sobre las bandas crudas.')
    else:
        print('-> El embedding AlphaEarth NO supera a las bandas '
              'crudas en este baseline tabular.')
else:
    print('Sin delta — comparativa omitida.')

Escenario ganador: AlphaEarth 64-dim
Delta F1-macro AlphaEarth - Sentinel-2 crudo: +0.1320
-> El embedding AlphaEarth aporta valor incremental sobre las bandas crudas.


## 8. Conclusiones

Este notebook construyo un punto de referencia para clasificar cultivos a partir de imagenes satelitales y lo sometio a tres preguntas: ¿que tan bien funciona un modelo de arboles sencillo?, ¿que caracteristicas explican sus predicciones?, y ¿el embedding AlphaEarth aporta algo frente a las bandas satelitales sin procesar? Lo que encontramos:

### ¿AlphaEarth aporta valor?

La comparativa de la seccion 7 da una respuesta con datos. El **embedding AlphaEarth** es una representacion compacta de 64 numeros que resume un ano de observaciones satelitales; las **bandas Sentinel-2 crudas** son los 10 canales del satelite promediados. La diferencia de F1-macro entre ambos escenarios indica si ese resumen aprendido aporta informacion que el promedio simple de las bandas pierde.

- Si AlphaEarth supera a las bandas crudas, el resumen aprendido captura senal multisensor y estacional que el promedio destruye.
- Si quedan empatados, ambas representaciones son equivalentes para un modelo de arboles a nivel de parcela.
- Si las bandas crudas ganan, el problema no esta en la representacion sino en haber promediado el tiempo: la solucion es usar la serie temporal completa.

### Hallazgos

1. **El techo de este modelo es estructural, no de ajuste.** Las curvas de aprendizaje (seccion 5b) muestran que el modelo no sobreajusta: simplemente ha llegado a su capacidad maxima sobre datos que ya perdieron la dimension temporal. Anadir mas arboles o mas profundidad no movera ese techo.
2. **La representacion de los datos importa mas que el algoritmo.** Random Forest y XGBoost rinden parecido dentro de cada escenario; la diferencia grande de desempeno aparece **entre escenarios**. La pregunta clave no es que clasificador usar, sino como representar la evolucion del cultivo en el tiempo.
3. **Promediar el tiempo es el cuello de botella.** Los tres escenarios resumen el ano en un solo vector. Pero cultivos espectralmente parecidos solo se distinguen por **como cambian a lo largo de la temporada** — y esa trayectoria se pierde al promediar.

### Lo que sigue

- **Modelos que usen la serie temporal completa.** Este baseline fija el piso de desempeno; los modelos siguientes deben procesar la secuencia de imagenes Sentinel-2 mes a mes — no su promedio anual — para captar la estacionalidad que separa cultivos parecidos. Si aun asi no superan estas cifras, el limite estaria en los datos, no en el modelo.
- **AlphaEarth como caracteristica de apoyo.** El embedding se incorporara como una entrada mas al combinar varios modelos, no como sustituto de la serie temporal cruda.
- **Mismo protocolo de evaluacion.** La validacion cruzada espacial con zona de exclusion entre parcelas vecinas se mantiene en las fases siguientes, para que las cifras sean comparables entre experimentos.

El baseline cumple su proposito: es **honesto, interpretable y reproducible** — establece el piso de desempeno, documenta sus propias limitaciones y deja un protocolo de evaluacion y una metrica principal que el resto del proyecto puede heredar.